# Automatización WB — Fase 1 (Demanda) + Fase 2 (Instructores) — hasta escribir "LCK B767"/"LCK B787" en la Matriz

Equivalente al notebook NB (`Automatizacion_Fase1_2_Demanda_Instructores.ipynb`), pero para las
flotas WB **B767** y **B787**. Son **dos procesos separados**: corres este mismo notebook una vez
con `FLOTA_WB = "767"` y, por separado, otra vez con `FLOTA_WB = "787"` (celda de configuración,
sección 3).

Cubre, en **modo VISTA PREVIA** (no escribe todavía en tus Google Sheets reales):

1. **Archivo 9** (pestaña "LCK 767" o "LCK 787") → contar tripulantes con `PROGRAMAR = Sí`.
2. **Rol Instructores LP OCTUBRE** → filtrar instructores con `IDE B767 = OK` o `IDE B787 = OK`
   (confirmado por Fernando: mismo archivo que usa NB, solo cambia el nombre de columna).
3. Calcular bloques (pairings) necesarios y vuelos a buscar.
4. **Matriz de Rol de Instructores** (la MISMA pestaña que usa NB — confirmado por Fernando; los
   instructores WB deben existir como filas nuevas ahí, agregadas a mano igual que se hizo con
   los de NB) → repartir esos bloques entre los instructores IDE de la flota elegida: variedad
   (round-robin), sin 2 días consecutivos para el mismo instructor, solo lunes-viernes.
5. Mostrar la vista previa (quién, qué día) **sin escribir nada todavía**.

**Punto abierto importante — confirmado con Fernando pero resuelto de forma diferida:** para
B767 con ruta LIM-MIA-LIM, el pairing real tiene la ida y la vuelta en fechas distintas (viaje
de varios días), y Fernando confirmó que en la Matriz se deben marcar **ambos días** (ida y
vuelta) para el instructor. Pero la separación exacta entre esas dos fechas depende del pairing
real que se encuentre en BigQuery (varía por pairing) y B767 también vuela la ruta LIM-SCL-LIM
(vuelta el mismo día, igual que B787) — este notebook (reparto genérico, todavía sin datos
reales de vuelos) **no puede saber cuál de los dos casos va a tocar cada bloque**. Por eso acá
se reserva **solo 1 celda** (el día de ida) por bloque; la segunda celda (día de vuelta, solo
si el pairing real resulta ser MIA) se agrega en la Fase 3/4, cuando ya se conoce la fecha real
— ver la sección final de este notebook y el notebook de Fase 3/4 de WB.


## 1. Instalar dependencias y autenticar

In [ ]:
!pip install -q gspread


In [ ]:
import datetime
import math
from collections import Counter
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("Autenticado y autorizado para leer/escribir Google Sheets con tu cuenta.")


## 2. Elegir la flota WB

Cambia `FLOTA_WB` a `"767"` o `"787"` y vuelve a correr todo el notebook para la otra flota
(son procesos separados, cada uno con su propio Archivo 9 / Rol / Archivo 10, pero comparten
la misma Matriz).

Cupos por vuelo (manual "Parte 2", reglas transversales, confirmado 2026-09-24): **B767 = 1 TJ +
4 TC (5 cupos)**, **B787 = 6 tripulantes por vuelo (6 cupos)**. La "capacidad por bloque" (2
vuelos = 1 pairing ida+vuelta, igual patrón que NB) es una extensión directa de esa regla, no
un dato confirmado aparte — revisar la primera vez que corra contra pairings reales.

In [ ]:
FLOTA_WB = "767"  # o "787" -> cambiar y volver a correr todo el notebook para la otra flota

WB_CONFIG = {
    "767": {
        "archivo9_gid": 1437673054,
        "archivo9_tab_esperada": "LCK 767",
        "rol_col_ide": "IDE B767",
        "archivo10_gid": 1025385274,
        "archivo10_tab_esperada": "prueba de LCK 767",
        "cupos_por_vuelo": 5,          # manual: B767 = 1 TJ + 4 TC
        "actividad_matriz": "LCK B767",
    },
    "787": {
        "archivo9_gid": 1122326179,
        "archivo9_tab_esperada": "LCK 787",
        "rol_col_ide": "IDE B787",
        "archivo10_gid": 16355364,
        "archivo10_tab_esperada": "prueba de LCK 787",
        "cupos_por_vuelo": 6,          # manual: B787 = 6 tripulantes/vuelo
        "actividad_matriz": "LCK B787",
    },
}

if FLOTA_WB not in WB_CONFIG:
    raise ValueError("FLOTA_WB debe ser '767' o '787'")

cfg = WB_CONFIG[FLOTA_WB]
print(f"Flota seleccionada: WB {FLOTA_WB}")
print(f"  Actividad a escribir en Matriz: '{cfg['actividad_matriz']}'")
print(f"  Columna IDE en Rol Instructores: '{cfg['rol_col_ide']}'")
print(f"  Cupos por vuelo: {cfg['cupos_por_vuelo']}")


## 3. Abrir las hojas

Archivo 9 y Archivo 10 son los MISMOS archivos que usa NB (solo cambia la pestaña / `gid` según
la flota elegida). Rol de Instructores y Matriz también son los mismos archivos que usa NB.

In [ ]:
URL_ARCHIVO_9 = "https://docs.google.com/spreadsheets/d/1d95aUJtNVAtHd3tECTcsc2WHM8b557Jbl1DcCDsJGuo/edit"
URL_ROL_INSTRUCTORES = "https://docs.google.com/spreadsheets/d/1yMvgb_O4qxpCCE4bAqD4XhW2GQI9ZObf2EAnymXaReE/edit?gid=1933640306"
URL_MATRIZ = "https://docs.google.com/spreadsheets/d/19WmwaoLDZnArNztu_dJNwi7bjrGq-_cx_gw0aN96zfk/edit?gid=580414308"
URL_ARCHIVO_10_WB = "https://docs.google.com/spreadsheets/d/1NZN565fOJUtoETQvvPzdHpRvyHp4stY2hsrqtjNjSEU/edit"

sh_archivo9 = gc.open_by_url(URL_ARCHIVO_9)
ws_archivo9 = sh_archivo9.get_worksheet_by_id(cfg["archivo9_gid"])

sh_rol_ins = gc.open_by_url(URL_ROL_INSTRUCTORES)
ws_rol_ins = sh_rol_ins.get_worksheet_by_id(1933640306)

sh_matriz = gc.open_by_url(URL_MATRIZ)
ws_matriz = sh_matriz.get_worksheet_by_id(580414308)  # misma pestaña que usa NB

sh_archivo10 = gc.open_by_url(URL_ARCHIVO_10_WB)
ws_archivo10_wb = sh_archivo10.get_worksheet_by_id(cfg["archivo10_gid"])

print("Hojas abiertas:", ws_archivo9.title, "|", ws_rol_ins.title, "|", ws_matriz.title, "|", ws_archivo10_wb.title)
if ws_archivo9.title != cfg["archivo9_tab_esperada"]:
    print(f"AVISO: Archivo 9 -> esperaba la pestaña '{cfg['archivo9_tab_esperada']}' pero el gid abrió '{ws_archivo9.title}'.")
if ws_archivo10_wb.title != cfg["archivo10_tab_esperada"]:
    print(f"AVISO: Archivo 10 -> esperaba la pestaña '{cfg['archivo10_tab_esperada']}' pero el gid abrió '{ws_archivo10_wb.title}'.")


## 4. Archivo 9 → demanda

Mismo patrón robusto que NB: se lee con `get_all_values()` (no `get_all_records()`) y se busca
la fila real de encabezados por contenido ("BP" + "PROGRAMAR"), sin depender de la columna/fila
exacta — así no importa que en tu captura "PROGRAMAR" aparezca en G4 (LCK 767) o H4 (LCK 787).

In [ ]:
valores_9 = ws_archivo9.get_all_values()

fila_header_idx = None
for i, fila in enumerate(valores_9):
    celdas_norm = [c.strip() for c in fila]
    if "BP" in celdas_norm and "PROGRAMAR" in celdas_norm:
        fila_header_idx = i
        break

if fila_header_idx is None:
    raise RuntimeError("No encontré una fila con 'BP' y 'PROGRAMAR' en Archivo 9 -> revisar estructura real de la hoja.")

encabezados_9 = valores_9[fila_header_idx]
col_bp = encabezados_9.index("BP")
col_programar = encabezados_9.index("PROGRAMAR")

filas_datos_9 = valores_9[fila_header_idx + 1:]
print(f"Fila de encabezado real detectada: fila {fila_header_idx + 1} (1-indexado)")
print(f"Filas de datos leídas de Archivo 9 ({cfg['actividad_matriz']}): {len(filas_datos_9)}")

bps_demanda = []
for fila in filas_datos_9:
    bp = fila[col_bp].strip().lstrip("'") if col_bp < len(fila) else ""
    programar = fila[col_programar].strip().lower() if col_programar < len(fila) else ""
    if not bp:
        continue
    if programar in ("si", "sí"):
        bps_demanda.append(bp)

demanda = len(bps_demanda)
print(f"Tripulantes con PROGRAMAR = Sí: {demanda}")


## 5. Rol de Instructores → quiénes son IDE de esta flota

Solo "OK" cuenta como IDE habilitado (mismo criterio confirmado para NB, extendido a WB).

In [ ]:
registros_rol = ws_rol_ins.get_all_records()
df_rol = pd.DataFrame(registros_rol)

col_ide = cfg["rol_col_ide"]
if col_ide not in df_rol.columns:
    raise RuntimeError(f"No encontré la columna '{col_ide}' en Rol Instructores -> columnas disponibles: {list(df_rol.columns)}")

instructores_ide_df = df_rol[df_rol[col_ide].astype(str).str.strip().str.upper() == "OK"].copy()
instructores_ide = list(zip(instructores_ide_df.iloc[:, 0].astype(str), instructores_ide_df["Nombre"]))

print(f"Instructores {col_ide} = OK: {len(instructores_ide)}")
for bp, nombre in instructores_ide:
    print(f"  {bp}  {nombre}")


## 6. Calcular bloques (pairings) necesarios y vuelos a buscar

Un "bloque" = 1 pairing (ida + vuelta = 2 vuelos), igual patrón que NB ("1 día-IDE = 2 vuelos").
`CAPACIDAD_TC_POR_BLOQUE = cupos_por_vuelo × 2` — ver advertencia de la sección 2 sobre esta
fórmula.

In [ ]:
CAPACIDAD_TC_POR_BLOQUE = cfg["cupos_por_vuelo"] * 2

bloques_necesarios = math.ceil(demanda / CAPACIDAD_TC_POR_BLOQUE) if demanda else 0
vuelos_a_buscar = bloques_necesarios * 2

print(f"Demanda (tripulantes a chequear, WB {FLOTA_WB}): {demanda}")
print(f"Cupos por vuelo: {cfg['cupos_por_vuelo']}")
print(f"Bloques (pairings) necesarios (demanda / {CAPACIDAD_TC_POR_BLOQUE}): {bloques_necesarios}")
print(f"Vuelos a buscar (bloques x 2 piernas): {vuelos_a_buscar}")


## 7. Leer la Matriz: fechas lunes-viernes y celdas ya ocupadas

Misma estructura y misma pestaña que usa NB (fila 2 = fechas desde columna C, datos desde fila
3). Los instructores WB deben existir como filas nuevas en esa misma hoja (agregadas a mano),
si no, `bp_a_fila_matriz` no los va a encontrar y se van a saltar con un aviso en la celda de
escritura.

In [ ]:
FILA_ENCABEZADO_FECHAS = 2   # fila con "BP","Nombre",fecha1,fecha2,... (1-indexado)
FILA_PRIMER_INSTRUCTOR = 3   # primera fila de datos (1-indexado)
COL_PRIMERA_FECHA = 3        # columna C (1-indexado)

valores = ws_matriz.get_all_values()

fila_fechas = valores[FILA_ENCABEZADO_FECHAS - 1]
fechas_matriz = []
for celda in fila_fechas[COL_PRIMERA_FECHA - 1:]:
    if not celda.strip():
        fechas_matriz.append(None)
        continue
    try:
        fechas_matriz.append(datetime.datetime.strptime(celda.strip(), "%d/%m/%Y").date())
    except ValueError:
        fechas_matriz.append(None)

fechas_lunes_viernes = [f for f in fechas_matriz if f is not None and f.weekday() < 5]
print(f"Fechas totales en la matriz: {sum(1 for f in fechas_matriz if f is not None)}")
print(f"De esas, lunes-viernes: {len(fechas_lunes_viernes)}")

celdas_ocupadas = set()
filas_datos = valores[FILA_PRIMER_INSTRUCTOR - 1:]
bp_a_fila_matriz = {}
for i, fila in enumerate(filas_datos):
    if not fila or not fila[0].strip():
        continue
    bp = fila[0].strip()
    bp_a_fila_matriz[bp] = FILA_PRIMER_INSTRUCTOR + i
    for j, celda in enumerate(fila[COL_PRIMERA_FECHA - 1:]):
        if celda.strip():
            fecha = fechas_matriz[j] if j < len(fechas_matriz) else None
            if fecha is not None:
                celdas_ocupadas.add((bp, fecha))

print(f"Celdas ya ocupadas por otra actividad (no se van a tocar): {len(celdas_ocupadas)}")

instructores_sin_fila = [nombre for bp, nombre in instructores_ide if bp not in bp_a_fila_matriz]
if instructores_sin_fila:
    print(f"AVISO: {len(instructores_sin_fila)} instructor(es) IDE {FLOTA_WB} no tienen fila en la Matriz todavía (agrégalos a mano antes de escribir): {instructores_sin_fila}")


## 8. Algoritmo de reparto

Mismo algoritmo que NB (round-robin, variedad, sin 2 días consecutivos por instructor, solo
lunes-viernes, respeta celdas ya ocupadas). Reserva **1 celda por bloque** (el día de ida) —
ver la nota del encabezado sobre por qué la eventual segunda celda de vuelta (solo para pairings
B767-MIA) se resuelve en la Fase 3/4 y no acá.

In [ ]:
def asignar_lck_round_robin(instructores_ide, fechas_lunes_viernes, bloques_necesarios,
                            celdas_ocupadas=None):
    celdas_ocupadas = set(celdas_ocupadas or set())
    DIAS_ES = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

    asignaciones = []
    dias_usados_por_instructor = {bp: set() for bp, _ in instructores_ide}
    n_instructores = len(instructores_ide)
    n_fechas = len(fechas_lunes_viernes)

    if n_instructores == 0:
        raise RuntimeError("No hay instructores IDE para esta flota -> revisar Rol de Instructores.")
    if n_fechas == 0:
        raise RuntimeError("No hay fechas lunes-viernes en la Matriz -> revisar estructura de la hoja.")

    idx_fecha = 0
    idx_instructor = 0

    for _ in range(bloques_necesarios):
        asignado = False
        for _intento_fecha in range(n_fechas):
            fecha = fechas_lunes_viernes[idx_fecha % n_fechas]
            idx_fecha += 1

            for _intento_instr in range(n_instructores):
                bp, nombre = instructores_ide[idx_instructor % n_instructores]
                idx_instructor += 1

                if (bp, fecha) in celdas_ocupadas:
                    continue
                if fecha in dias_usados_por_instructor[bp]:
                    continue
                adyacente = any(
                    abs((fecha - otra).days) == 1
                    for otra in dias_usados_por_instructor[bp]
                )
                if adyacente:
                    continue

                dias_usados_por_instructor[bp].add(fecha)
                celdas_ocupadas.add((bp, fecha))
                asignaciones.append((bp, nombre, fecha, DIAS_ES[fecha.weekday()]))
                asignado = True
                break
            if asignado:
                break
        if not asignado:
            raise RuntimeError("No se encontró ningún instructor/fecha libre -> faltan instructores IDE o días hábiles disponibles")

    return asignaciones


asignaciones = asignar_lck_round_robin(instructores_ide, fechas_lunes_viernes, bloques_necesarios, celdas_ocupadas)
print(f"Slots asignados: {len(asignaciones)} de {bloques_necesarios} necesarios")


## 9. Vista previa (todavía NO se escribe nada en tus Sheets reales)

In [ ]:
df_preview = pd.DataFrame(asignaciones, columns=["BP", "Instructor", "Fecha", "Día"])
df_preview["Fecha"] = df_preview["Fecha"].apply(lambda d: d.strftime("%d/%m/%Y"))
df_preview = df_preview.sort_values(["Fecha", "Instructor"]).reset_index(drop=True)

print(f"=== VISTA PREVIA: dónde se escribiría '{cfg['actividad_matriz']}' ===")
display(df_preview)

conteo = Counter(a[1] for a in asignaciones)
print("\nReparto por instructor:")
for nombre, n in conteo.items():
    print(f"  {nombre}: {n}")

nombre_archivo = f"Vista_previa_{cfg['actividad_matriz'].replace(' ', '_')}.xlsx"
df_preview.to_excel(nombre_archivo, index=False)
print(f"\nGuardado para revisión: {nombre_archivo}")

print(f"\nBPs que se pegarían en Archivo 10, hoja '{ws_archivo10_wb.title}': {len(bps_demanda)}")


## 10. Escribir en los Sheets reales — DESACTIVADO por defecto

Revisa primero la vista previa; si se ve bien, descomenta y corre esta celda.

BP se pega desde la fila **4** (confirmado por Fernando para ambas pestañas WB — distinto de
NB, que empieza en fila 3).

In [ ]:
# --- DESCOMENTAR SOLO DESPUÉS DE REVISAR LA VISTA PREVIA ---

# (a) pegar BPs de la demanda en Archivo 10, empieza en A4
# rango_bp = f"A4:A{3 + len(bps_demanda)}"
# ws_archivo10_wb.format(rango_bp, {"numberFormat": {"type": "NUMBER", "pattern": "0"}})
# ws_archivo10_wb.update(
#     rango_bp,
#     [[int(bp)] for bp in bps_demanda],
#     value_input_option="USER_ENTERED",
# )
# print(f"Pegados {len(bps_demanda)} BP en '{ws_archivo10_wb.title}'.")

# (b) escribir la actividad en la Matriz, en las celdas calculadas (1 celda por bloque, día de ida)
# celdas_a_escribir = []
# for bp, nombre, fecha, _dia in asignaciones:
#     fila = bp_a_fila_matriz.get(bp)
#     if fila is None:
#         print(f"AVISO: no encontré la fila de {nombre} (BP {bp}) en la Matriz, se salta.")
#         continue
#     col_idx = COL_PRIMERA_FECHA + fechas_matriz.index(fecha)
#     celdas_a_escribir.append(gspread.Cell(row=fila, col=col_idx, value=cfg["actividad_matriz"]))
#
# if celdas_a_escribir:
#     ws_matriz.update_cells(celdas_a_escribir)
#     print(f"Escritas {len(celdas_a_escribir)} celdas '{cfg['actividad_matriz']}' en la Matriz.")


## 11. Estado y supuestos — sin inventar nada

### Lo que se automatiza aquí
- Demanda = conteo de `PROGRAMAR = Sí` en Archivo 9 (pestaña "LCK 767" o "LCK 787" según
  `FLOTA_WB`), con el mismo buscador robusto de encabezado por contenido que NB.
- Filtro de instructores con `IDE B767 = OK` / `IDE B787 = OK` en Rol de Instructores (mismo
  archivo que NB, confirmado por Fernando — solo cambia el nombre de columna).
- Bloques necesarios = demanda / (cupos_por_vuelo × 2), vuelos a buscar = bloques × 2.
- Reparto round-robin en la MISMA Matriz que usa NB, sin días consecutivos, solo
  lunes-viernes, respetando celdas ya ocupadas — reserva 1 celda por bloque (día de ida).

### Confirmado directamente por Fernando (2026-09-24)
- Rol de Instructores WB = el mismo archivo "Rol Instructores LP OCTUBRE" que usa NB, columna
  `IDE B767` / `IDE B787` en vez de `IDE A320`.
- La Matriz para WB va en la **misma pestaña** que usa NB (filas nuevas para los instructores
  WB), no en una pestaña separada.
- Para B767, con pairings LIM-MIA-LIM (ida y vuelta en fechas distintas), la Matriz debe marcar
  **ambos días** para el instructor — implementado en la Fase 3/4 (ver más abajo), no acá.
- Los grupos de instructores (Grupo 1/2/3 por flota) sí varían mes a mes, igual que en NB —
  se van a leer en vivo desde celdas editables en Archivo 10, no hardcodeados (mismo patrón
  que `AD2`/`AE2`/`AF2`/`AG2` de NB).

### Supuestos derivados que hay que confirmar la primera vez que corra contra los Sheets reales
- **Cupos por vuelo** (B767 = 5, B787 = 6) vienen del manual ("Dotación por flota"), pero la
  fórmula "capacidad por bloque = cupos × 2" es una EXTENSIÓN del patrón de NB, no una regla
  confirmada aparte para WB — revisar contra la demanda real del primer mes.
- Nombres exactos de columna: `BP`, `PROGRAMAR` (Archivo 9); columna IDE, `Nombre` (Rol
  Instructores) — igual que NB, si el encabezado real difiere hay que ajustar.
- Estructura de la Matriz (fila 2 = fechas desde columna C, fila 3+ = instructores) asumida
  igual a la de NB, porque es el mismo archivo/pestaña.

### Punto abierto — resuelto de forma diferida, no acá
Para B767, este notebook reserva **solo 1 celda** (día de ida) por bloque, aunque Fernando
confirmó que un pairing MIA necesita 2 celdas (ida y vuelta). No se puede reservar la segunda
celda acá porque (a) la separación exacta de días entre ida y vuelta depende del pairing real
que se encuentre en BigQuery, y (b) B767 también vuela la ruta LIM-SCL-LIM (vuelta el mismo
día, sin necesidad de una segunda celda) y este notebook, antes de buscar vuelos reales, no
tiene forma de saber cuál de los dos casos le va a tocar a cada bloque. La segunda celda (solo
cuando corresponda) se agrega en el notebook de Fase 3/4 de WB, en el momento en que ya se
conoce la fecha real de vuelta.

### Lo que sigue sin automatizar (igual que NB)
- `OBS FREEZE` (Archivo 9), `INS F a considerar` y `Grupo` (Archivo 10) — confirmado que los
  completa la otra persona del proceso / se resuelven en la Fase 4.
- El descanso reglamentario se respeta indirectamente (no 2 LCK consecutivos al mismo
  instructor), pero no se valida contra otros vuelos reales que el instructor ya tenga
  asignados fuera de esta Matriz.
